# 01 · Data preparation

**Project:** protein language model (pLM) embeddings and sequence–affinity relationships in
LxCxE–pRb interactions.

This is the **only notebook that reads the raw experimental table**. It cleans the SIMBA
deep mutational scanning (DMS) data once and stores two tidy tables that every other notebook reuses.

| Input (`data/raw/`) | Output (`data/processed/`) |
|---|---|
| `lib1_and_2_B44-B45_Rb-pep_PEPTIDES_DATA.xlsx` (sheet `ALL_PEPTIDES_scores_z_SD`) | `df_tp.csv` – wild-type (WT) peptides, one row per protein |
| | `df_dms.csv` – point mutants with their ΔZ-score |

**Interaction metric.** For every mutant the change in interaction strength is

$$\Delta Z = Z_{\mathrm{mut}} - Z_{\mathrm{WT}}$$

computed for the two SIMBA constructs: `pm_rb` (column `C_Pm-Rb_SIMBA-score`, the main analysis) and
`pw_rb` (column `F_Pw-Rb_SIMBA-score`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Project paths
#   * Locally: start Jupyter from the `notebooks/` folder.
#   * Google Colab: Google Drive is mounted; edit the folder name below so it
#     points to the project folder stored in your Drive.
#   * Anywhere: set the PROJECT_DIR environment variable to override.
# ---------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/plm-embeddings-lxcxe-prb")
except ImportError:
    DEFAULT_PROJECT_DIR = Path.cwd().parent

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", DEFAULT_PROJECT_DIR))
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
for folder in (PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_figure(name):
    """Save the current matplotlib figure into figures/ (call before plt.show())."""
    plt.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")

import re

## 1. Configuration

In [ ]:
RAW_XLSX = RAW_DIR / "lib1_and_2_B44-B45_Rb-pep_PEPTIDES_DATA.xlsx"
SHEET_NAME = "ALL_PEPTIDES_scores_z_SD"

# Peptides containing the LxCxE motif that are analysed in this project
TARGET_PROTEINS = [
    "ARI4A_HUMAN",
    "KDM5A_HUMAN",
    "HDAC1_HUMAN",
    "VE7_HPV16",
    "E1A_ADE05_LxCxE",
]

# Raw column name -> clean column name
RAW_COLUMNS = {
    "Unique Peptide": "peptide",
    "Category": "category",
    "Description": "description",
    "C_Pm-Rb_SIMBA-score": "score_pm_rb",
    "F_Pw-Rb_SIMBA-score": "score_pw_rb",
}
SCORE_COLUMNS = ["score_pm_rb", "score_pw_rb"]

## 2. Load the raw table

In [ ]:
if not RAW_XLSX.exists():
    raise FileNotFoundError(
        f"Raw data not found: {RAW_XLSX}\n"
        "Place the SIMBA Excel file in data/raw/ (see data/README.md)."
    )

df_raw = pd.read_excel(RAW_XLSX, sheet_name=SHEET_NAME)
df = df_raw[list(RAW_COLUMNS)].rename(columns=RAW_COLUMNS)

# Rows whose description mentions one of the target proteins
protein_pattern = "|".join(TARGET_PROTEINS)
has_protein = df["description"].str.contains(protein_pattern, case=False, na=False)
print(f"{len(df)} rows in the table, {has_protein.sum()} mention a target protein")

## 3. Wild-type peptides (`df_tp`)

WT reference peptides are the `d_TP` rows flagged `NC-Flank`. Their description follows
`LXCXE|<UniProt ID>|<protein>|NC-Flank|<start>|NA|No|`.

In [ ]:
has_flank = df["description"].str.contains("NC-Flank", case=False, na=False)
df_tp = df[(df["category"] == "d_TP") & has_protein & has_flank].copy()

wt_info = df_tp["description"].str.extract(
    r"^LXCXE\|(?P<uniprot_id>[^|]+)\|(?P<protein>[^|]+)\|"
)
if wt_info.isna().any().any():
    raise ValueError("Could not parse UniProt ID / protein name from a WT description.")

df_tp = pd.concat([wt_info, df_tp], axis=1)

# Exactly one WT peptide per target protein
found = df_tp["protein"].tolist()
if sorted(found) != sorted(TARGET_PROTEINS):
    raise ValueError(f"Expected one WT peptide per target protein, found: {found}")

df_tp = (
    df_tp.set_index("protein")
    .loc[TARGET_PROTEINS]
    .reset_index()[["protein", "uniprot_id", "peptide", "category", "description"] + SCORE_COLUMNS]
)
df_tp

## 4. Point mutants (`df_dms`)

Mutants are the `d_DMS - LXCXE` rows. The mutation code at the end of the description
(e.g. `ARI4A_HUMAN_C7D` → Cys7→Asp) is parsed into position / WT residue / mutant residue and
validated against the WT peptide.

In [ ]:
df_dms = df[(df["category"] == "d_DMS - LXCXE") & has_protein].copy()

parsed = df_dms["description"].str.extract(
    r"^(?P<protein>.+)_(?P<wt_aa>[A-Z])(?P<position>\d+)(?P<mut_aa>[A-Z])$"
)
valid = parsed["protein"].isin(TARGET_PROTEINS)
if (~valid).any():
    print(f"Warning: dropping {(~valid).sum()} rows with an unparsable mutation code")

df_dms = pd.concat([parsed, df_dms], axis=1)[valid].copy()
df_dms["position"] = df_dms["position"].astype(int)
df_dms["mutation"] = df_dms["wt_aa"] + df_dms["position"].astype(str) + df_dms["mut_aa"]

# Validate each mutant against the WT peptide of its protein
wt_peptide = df_tp.set_index("protein")["peptide"]


def rebuild_mutant(row):
    """WT peptide with the described substitution applied."""
    wt = wt_peptide[row["protein"]]
    pos = row["position"]
    if not 1 <= pos <= len(wt) or wt[pos - 1] != row["wt_aa"]:
        return None
    return wt[: pos - 1] + row["mut_aa"] + wt[pos:]


expected_peptide = df_dms.apply(rebuild_mutant, axis=1)
consistent = expected_peptide == df_dms["peptide"]
if (~consistent).any():
    print(f"Warning: dropping {(~consistent).sum()} rows inconsistent with the WT peptide:")
    print(df_dms.loc[~consistent, "description"].tolist())
df_dms = df_dms[consistent].copy()

## 5. ΔZ-score

In [ ]:
wt_scores = df_tp.set_index("protein")[SCORE_COLUMNS]

for col in SCORE_COLUMNS:
    delta_col = col.replace("score_", "delta_")
    df_dms[delta_col] = df_dms[col] - df_dms["protein"].map(wt_scores[col])

df_dms["uniprot_id"] = df_dms["protein"].map(df_tp.set_index("protein")["uniprot_id"])

df_dms = df_dms[
    ["protein", "uniprot_id", "mutation", "position", "wt_aa", "mut_aa",
     "peptide", "category", "description"]
    + SCORE_COLUMNS
    + ["delta_pm_rb", "delta_pw_rb"]
].reset_index(drop=True)
df_dms.head()

## 6. Sanity checks and export

In [ ]:
assert not df_dms.duplicated(["protein", "mutation"]).any(), "Duplicated mutations found"
assert df_dms[["delta_pm_rb", "delta_pw_rb"]].notna().all().all(), "Missing ΔZ values"

summary = df_dms.groupby("protein", sort=False).agg(
    n_mutants=("mutation", "size"),
    positions=("position", lambda s: f"{s.min()}-{s.max()}"),
    mean_delta_pm_rb=("delta_pm_rb", "mean"),
)
summary.insert(0, "wt_peptide", df_tp.set_index("protein")["peptide"])
summary

In [ ]:
df_tp.to_csv(PROCESSED_DIR / "df_tp.csv", index=False)
df_dms.to_csv(PROCESSED_DIR / "df_dms.csv", index=False)
print(f"Saved df_tp.csv ({len(df_tp)} rows) and df_dms.csv ({len(df_dms)} rows) to {PROCESSED_DIR}")